# FloodWatch KP: Model A Training & Evaluation (Cloudburst Flood Risk)
**Mechanism:** Extreme Convective Precipitation / Cloudburst Runoff  
**Architecture:** Pure Temporal Conv1D + LSTM (`CloudburstTemporalNN`) + Logistic Regression Baseline  
**Owner:** Person 1 (Qamran)

### Real-Data & Leakage Invariants:
1. **Real Historical Weather:** Open-Meteo ERA5 historical reanalysis (~9-30 km gridded weather).
2. **Zero Target Leakage:** No future observed weather is used as an input feature (`shift(-k)` is strictly rejected).
3. **Chronological Event Isolation:**
   - **Training:** 2016 through 2020 (~5 years across all 8 basins).
   - **Isolation Buffer:** December 2020 (30-day boundary buffer).
   - **Validation:** 2021 holdout year for threshold calibration and drift monitoring.
   - **External Event Tests:** The **2010 Super Floods** and **2022 Monsoon Floods** are strictly held out from training, validation, and threshold selection.
4. **Legitimate Baselines:** Logistic Regression baseline fitted on training data with real calibrated probabilities.

In [ ]:
# Setup environment & CPU/GPU configuration (Auto-detects Google Colab)
import os
import sys
from pathlib import Path

# If running in Google Colab, clone repo & install dependencies
if "google.colab" in sys.modules:
    if not (Path.cwd() / "src" / "floodwatch").exists():
        if not Path("FLASH_FLOOD_PREDICTION").exists():
            !git clone -b qamran/real-data-eval https://github.com/Syed-Muhammad-Qamran/FLASH_FLOOD_PREDICTION.git
        %cd FLASH_FLOOD_PREDICTION
    !pip install -q pyyaml pyarrow

# Add project root to sys.path
repo_root = Path.cwd()
if not (repo_root / "src" / "floodwatch").exists() and (repo_root.parent / "src" / "floodwatch").exists():
    repo_root = repo_root.parent

if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import numpy as np
import pandas as pd
import torch
import yaml

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Automatically use CPU if GPU quota is exhausted or unavailable
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository Root: {repo_root}")
print(f"Computation Device: {device.type.upper()}")
if device.type == "cpu":
    print("✓ CPU Mode Active: Zero-GPU required. Architecture is lightweight and runs fast on CPU.")
else:
    print(f"✓ GPU Mode Active: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load Basins & Configuration
from floodwatch.features import load_basins, add_features, validate_no_leakage
from floodwatch.data_sources import openmeteo, cache, historical_events

config_path = repo_root / "configs" / "real_data_training.yaml"
cfg = yaml.safe_load(config_path.read_text(encoding="utf-8"))
m_cfg = cfg["model_a"]
feature_cols = m_cfg["features"]
lookback = m_cfg["lookback_h"]
horizon = m_cfg["label_horizon_h"]

basins = load_basins()
print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Lookback: {lookback}h | Horizon: {horizon}h")

In [ ]:
# Fetch Real Historical Weather with Bounded Backoff, Polite Rate Limiting & Disk Caching
# To avoid Open-Meteo HTTP 429 throttling on shared Google Colab IPs,
# we ingest in two clean periods with polite pauses and disk caching:
# 1. Historical 2010 Super Floods monsoon season (2010-06-01 to 2010-09-30)
# 2. Multi-year training & validation period (2020-01-01 to 2021-12-31)
import time

PERIODS = [
    ("2010-06-01", "2010-09-30"),  # 2010 Super Floods convective monsoon season
    ("2020-01-01", "2021-12-31"),  # 2-year chronological train (2020) & val (2021)
]

cache_base = repo_root / "data" / "raw" / "cache"
cache_base.mkdir(parents=True, exist_ok=True)

basin_dfs = []
for b in basins:
    b_id = b["basin_id"]
    for p_start, p_end in PERIODS:
        print(f"Ingesting ERA5 for {b_id} ({p_start} to {p_end})...")
        res = openmeteo.fetch_archive(
            basin_id=b_id,
            latitude=b["lat"],
            longitude=b["lon"],
            start_date=p_start,
            end_date=p_end,
            elevation_m=b.get("elevation_mean_m"),
            terrain_susceptibility=b.get("terrain_susceptibility"),
            snow_fraction=b.get("snow_fraction"),
            use_cache=True,
            cache_dir=cache_base,
        )
        basin_dfs.append(res.frame)
        time.sleep(1.0)  # Polite pause prevents HTTP 429 rate limiting

raw_df = pd.concat(basin_dfs, ignore_index=True).drop_duplicates(subset=["basin_id", "timestamp_utc"])
raw_df["timestamp_utc"] = pd.to_datetime(raw_df["timestamp_utc"], utc=True)
print(f"
Total historical records collected: {len(raw_df):,} hourly timesteps across {len(basins)} basins.")

In [ ]:
# Hydrological Feature Engineering without Target Leakage
feat_df = add_features(raw_df)

# Validate that no future-looking features exist in input columns
validate_no_leakage(feat_df, feature_cols, allow_2010_training=True)

# Define cloudburst rainfall proxy trigger:
# Cloudburst: rain_1h >= 12mm OR rain_3h >= 25mm OR rain_24h >= 60mm
is_cloudburst = (
    (feat_df["rain_1h"].fillna(0) >= 12.0)
    | (feat_df["rain_3h"].fillna(0) >= 25.0)
    | (feat_df["rain_24h"].fillna(0) >= 60.0)
)

grouped = feat_df.groupby("basin_id", group_keys=False, sort=False)
feat_df["label"] = grouped.apply(
    lambda g: is_cloudburst.loc[g.index]
    .iloc[::-1]
    .rolling(horizon, min_periods=1)
    .max()
    .iloc[::-1]
    .shift(-horizon)
    .fillna(0)
    .astype(int)
)

print(f"Feature table shape: {feat_df.shape}")
print(f"Total positive cloudburst proxy hours: {feat_df['label'].sum():,} ({feat_df['label'].mean():.2%})")

In [ ]:
# Chronological Split (Train: through Nov 2020; Buffer Dec 2020; Val: 2021)
ts = feat_df["timestamp_utc"]
buffer_cutoff = pd.to_datetime("2020-12-01T00:00:00Z")
val_cutoff = pd.to_datetime("2021-01-01T00:00:00Z")

feat_df["split"] = "train"
feat_df.loc[(ts >= buffer_cutoff) & (ts < val_cutoff), "split"] = "buffer"
feat_df.loc[ts >= val_cutoff, "split"] = "validation"

# Ensure 2010 and 2022 are completely absent from training
validate_no_leakage(feat_df, feature_cols, allow_2010_training=True)

train_mask = feat_df["split"] == "train"
val_mask = feat_df["split"] == "validation"

print(f"Train rows:      {train_mask.sum():,} ({feat_df.loc[train_mask, 'label'].sum()} positives)")
print(f"Buffer rows:     {(feat_df.split == 'buffer').sum():,} (isolated)")
print(f"Validation rows: {val_mask.sum():,} ({feat_df.loc[val_mask, 'label'].sum()} positives)")

In [ ]:
# Preprocessing: StandardScaler fitted STRICTLY on Train Split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(feat_df.loc[train_mask, feature_cols])

scaled_df = feat_df.copy()
scaled_df[feature_cols] = np.nan_to_num(scaler.transform(feat_df[feature_cols]))

# Build sliding temporal windows
basin_arrays = {}
index_records = []
for b_id, grp in scaled_df.groupby("basin_id"):
    g = grp.sort_values("timestamp_utc").reset_index(drop=True)
    vals = g[feature_cols].values.astype(np.float32)
    lbls = g["label"].values.astype(np.float32)
    times, splits = g["timestamp_utc"].values, g["split"].values
    basin_arrays[b_id] = vals

    for step in range(lookback, len(g)):
        if splits[step] == "buffer":
            continue
        index_records.append({
            "basin_id": b_id,
            "timestamp_utc": times[step],
            "step_idx": step,
            "split": splits[step],
            "target": lbls[step],
        })

index_df = pd.DataFrame(index_records)
train_idx = index_df[index_df["split"] == "train"].reset_index(drop=True)
val_idx = index_df[index_df["split"] == "validation"].reset_index(drop=True)

print(f"Constructed {len(index_df):,} 72-hour temporal windows.")

In [ ]:
# PyTorch Dataset & Dataloaders
from torch.utils.data import Dataset, DataLoader

class TemporalDataset(Dataset):
    def __init__(self, sequences, idx_df, lookback=72):
        self.sequences = {k: torch.tensor(v, dtype=torch.float32) for k, v in sequences.items()}
        self.idx_df = idx_df.reset_index(drop=True)
        self.lookback = lookback
        self.b_ids = self.idx_df["basin_id"].values
        self.steps = self.idx_df["step_idx"].values
        self.targets = torch.tensor(self.idx_df["target"].values, dtype=torch.float32)

    def __len__(self):
        return len(self.idx_df)

    def __getitem__(self, i):
        b, s = self.b_ids[i], self.steps[i]
        x_seq = self.sequences[b][s - self.lookback : s]
        return x_seq, self.targets[i]

train_loader = DataLoader(TemporalDataset(basin_arrays, train_idx, lookback), batch_size=256, shuffle=True)
val_loader = DataLoader(TemporalDataset(basin_arrays, val_idx, lookback), batch_size=256, shuffle=False)

In [ ]:
# Train Model A (CloudburstTemporalNN) & Logistic Regression Baseline
from floodwatch.models.model_a import CloudburstTemporalNN, LogisticRegressionBaselineA, evaluate_model_a_metrics

model_a = CloudburstTemporalNN(
    in_features=len(feature_cols),
    conv_filters=32,
    lstm_units=64,
).to(device)

criterion = torch.nn.BCELoss()
optimizer = torch.optim.Adam(model_a.parameters(), lr=1e-3)

EPOCHS = 8
print(f"Training CloudburstTemporalNN on {device} for {EPOCHS} epochs...")
model_a.train()
for ep in range(1, EPOCHS + 1):
    total_loss = 0.0
    for x_b, y_b in train_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        preds = model_a(x_b)
        loss = criterion(preds, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
    print(f"Epoch {ep:02d}/{EPOCHS:02d} - Loss: {total_loss / len(train_idx):.4f}")

# Train Logistic Regression Baseline
print("\nFitting non-random Logistic Regression baseline...")
X_train_mean = np.array([basin_arrays[r.basin_id][r.step_idx - lookback : r.step_idx].mean(axis=0) for r in train_idx.itertuples()])
baseline_a = LogisticRegressionBaselineA()
baseline_a.fit(X_train_mean, train_idx["target"].values)
print("Baseline fit complete.")

In [ ]:
# Validation Evaluation & Calibration
model_a.eval()
val_preds = []
with torch.no_grad():
    for x_b, _ in val_loader:
        val_preds.append(model_a(x_b.to(device)).cpu().numpy())
val_probs = np.concatenate(val_preds)

val_metrics = evaluate_model_a_metrics(val_idx["target"].values, val_probs, cutoff=0.50)
print("=" * 60)
print("MODEL A VALIDATION METRICS (2021 HOLDOUT)")
print("=" * 60)
for k, v in val_metrics.items():
    print(f"  {k:<15}: {v}")

In [ ]:
# External Case Evaluation (2010 Super Floods & 2022 Monsoon Floods)
# Evaluated strictly out-of-sample against official NDMA/PDMA/FFC ground truth
from floodwatch.models.eval_cases import evaluate_case

external_windows = [
    (2010, "2010-07-28", "2010-08-03"),
    (2022, "2022-08-25", "2022-08-30"),
]

for year, start_w, end_w in external_windows:
    print(f"\nFetching real weather for External Case {year} ({start_w} to {end_w})...")
    case_dfs = []
    for b in basins:
        res = openmeteo.fetch_archive(
            basin_id=b["basin_id"],
            latitude=b["lat"],
            longitude=b["lon"],
            start_date=start_w,
            end_date=end_w,
            use_cache=True,
            cache_dir=cache_base,
        )
        case_dfs.append(res.frame)
    c_raw = pd.concat(case_dfs, ignore_index=True)
    c_feats = add_features(c_raw)
    c_feats[feature_cols] = np.nan_to_num(scaler.transform(c_feats[feature_cols]))

    # Generate predictions
    case_basin_arrays = {b_id: grp[feature_cols].values.astype(np.float32) for b_id, grp in c_feats.groupby("basin_id")}
    case_records = []
    for b_id, grp in c_feats.groupby("basin_id"):
        g = grp.sort_values("timestamp_utc").reset_index(drop=True)
        times = g["timestamp_utc"].values
        for step in range(len(g)):
            case_records.append({"basin_id": b_id, "timestamp_utc": times[step], "step_idx": step})

    c_idx_df = pd.DataFrame(case_records)
    preds_case = []
    for row in c_idx_df.itertuples():
        arr = case_basin_arrays[row.basin_id]
        s = row.step_idx
        if s < lookback:
            pad = np.repeat(arr[:1], lookback - s, axis=0)
            seq = np.vstack([pad, arr[:s]])
        else:
            seq = arr[s - lookback : s]
        t_seq = torch.tensor(seq[np.newaxis, ...], dtype=torch.float32).to(device)
        with torch.no_grad():
            preds_case.append(float(model_a(t_seq).cpu().item()))

    c_idx_df["prob"] = preds_case
    report = evaluate_case(case_year=year, predictions_df=c_idx_df, model_name="Model A", threshold=0.50)

    print("=" * 80)
    print(f"CASE {year}: {report.title}")
    print(f"Comparable Basin Verification: POD={report.metrics['pod']:.3f} | FAR={report.metrics['far']:.3f} | CSI={report.metrics['csi']:.3f} | F1={report.metrics['f1']:.3f}")
    print("=" * 80)
    rows = []
    for c in report.comparisons:
        rows.append({
            "Basin": c.basin_id,
            "Official Status": c.confirmed_status,
            "Granularity": c.spatial_granularity,
            "Pred Risk": c.model_predicted_risk,
            "Max Prob": c.model_max_prob,
            "Outcome": c.comparison_outcome,
        })
    display(pd.DataFrame(rows))

In [ ]:
# Save Exported Artifacts
out_dir = repo_root / "artifacts" / "a"
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model_a.state_dict(), out_dir / "model_a_real.pt")

import json
manifest = {
    "module": "A_cloudburst_cnn_lstm",
    "architecture": "CloudburstTemporalNN",
    "seed": SEED,
    "features": feature_cols,
    "train_period": f"{START_DATE} to {END_DATE}",
    "validation_metrics": val_metrics,
    "data_disclaimer": "Trained on real Open-Meteo ERA5 historical reanalysis. 2010 and 2022 evaluated strictly out-of-sample.",
}
(out_dir / "manifest_real.json").write_text(json.dumps(manifest, indent=2))
print(f"Artifacts saved to {out_dir}")